# Example: Phenotyping API Usage

**Before you start:** Run from `Notebooks/examples/`. Replace `path/to/images` and the path template; set `mask_dir` if you want mask-based property presets.

**What this notebook does:** (1) discover images and metadata, (2) optionally audit then embed images, (3) optionally compute **hand-crafted** properties (needs masks for presets like `basic`), (4) distances and plots vs a reference group, (5) save results and HTML report.

Uses `get_embeddings()` for lazy-safe access when using checkpoints. See [Pipeline API](../../docs/reference/api/pipeline.md) and [Common Workflows](../../docs/examples/workflows.md).

**Author:** Aitor González-Marfil ([@AAitorG](https://github.com/AAitorG))

---

## 1. Setup

In [ ]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from phenome import PhenoMe, get_metadata_from_path, load_dinov2_model

## 2. Load Data with Metadata

Loads the vision model and registers all image paths (and optional mask paths). Use `get_metadata_from_path` when folders encode condition, e.g. `.../(condition)/(filename).*`.

In [ ]:
model, wrapper = load_dinov2_model(model_name="dinov2_vitb14_reg")
pheno = PhenoMe(seed=42)

metadata_fn = get_metadata_from_path(".../(condition)/(filename).*")
image_dir = "path/to/images"
mask_dir = "path/to/masks"  # Set to a real path when you have masks; else None

file_df = pheno.find_files(
    image_dir,
    metadata_fn=metadata_fn,
    mask_dir=mask_dir if mask_dir and os.path.exists(mask_dir) else None,
)

print(f"Found {len(file_df)} images")
print(f"Conditions: {file_df['condition'].unique().tolist()}")

## 3. Audit and Process

`audit_data()` summarizes file shapes (see auditing tutorial). `process_images()` runs the neural network to produce **embeddings**. Use `channel_mode='split'` for multi-channel fluorescence.

In [ ]:
audit_df = pheno.audit_data()
pheno.process_images(wrapper, channel_mode="split")

# Preset 'basic' includes mask-based features — run only when masks were linked in find_files
has_masks = "mask_path" in file_df.columns and file_df["mask_path"].notna().any()
if has_masks:
    pheno.compute_properties(property_preset="basic")
else:
    print("Skipping compute_properties: set mask_dir in find_files when you have segmentations.")

## 4. Access Embeddings

Use `pheno.get_embeddings()` for lazy-safe access when using checkpoints. Avoid `pheno.results.embeddings` with lazy loading.

In [ ]:
embeddings = pheno.get_embeddings()
if embeddings is not None:
    print(f"Embeddings shape: {embeddings.shape}")

## 5. Distance Analysis and Visualization

Use `distance_results` (keyword) for clarity. Pass `group_by` for violin plots.

In [ ]:
distance_results = pheno.compute_reference_distances(
    reference_filters={"condition": "Control"}, source="embeddings", mode="centroid"
)

pheno.plot_pca(color_by="condition")
pheno.plot_distance_distribution(distance_results, group_by="condition")

## 6. Save Results and Generate Report

Save checkpoint and optionally generate an HTML report.

**Clustering in the report:** With `include_clustering=True`, the report runs `compute_clustering` internally if no `cluster` labels exist yet. For **interactive** notebooks or scripts—tuning `n_clusters`, comparing to metadata, or using `find_prototypes` / `detect_outliers`—call those methods yourself; see [04_exploratory_analysis.ipynb](04_exploratory_analysis.ipynb).

In [ ]:
pheno.save_results(output_dir="results/")

pheno.generate_report(
    output_path="results/report.html",
    reference_filters={"condition": "Control"},
    include_clustering=True,
    include_image_gallery=True,
    color_by="condition",
)

## See Also

- [04_exploratory_analysis.ipynb](04_exploratory_analysis.ipynb): Clustering, prototypes, outliers, grouped property summaries, explainability
- [03_cell_painting_preset.ipynb](03_cell_painting_preset.ipynb): Cell Painting preset example
- [example_advanced_phenotyping.py](example_advanced_phenotyping.py): Full script-style pipeline (set your own data paths)
- [Report API](../../docs/reference/api/report.md): Report generation options
- [Report Guide](../../docs/guides/report-guide.md): How to read the HTML report
- [export_dataset_table](../../docs/reference/api/pipeline.md): Export metadata, properties, distances to CSV